# 01 - Exploracion de APIs

**Objetivo**: Probar cada API y ver que datos devuelven antes de construir el pipeline completo.

Vamos a probar:
1. **GeckoTerminal** - OHLCV, pools nuevos, datos de mercado
2. **DexScreener** - Buyers/sellers, boosts
3. **Solana RPC (Helius)** - Top holders, supply
4. **Etherscan** - Verificacion de contratos
5. **CoinGecko** - Precios historicos de BTC/ETH/SOL

> **Antes de empezar**: Copia `.env.example` a `.env` y llena tus API keys.

In [ ]:
# Configuracion inicial - ejecutar siempre primero
import sys
from pathlib import Path

# Agregar el directorio raiz del proyecto al path de Python
# Esto permite importar nuestros modulos (src.api, src.data, etc.)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Verificar que estamos en el directorio correcto
print(f"Directorio del proyecto: {PROJECT_ROOT}")
print(f"Archivos en raiz: {[f.name for f in PROJECT_ROOT.iterdir() if f.is_file()][:10]}")

In [ ]:
# Importar nuestros clientes de API
from src.api.coingecko_client import CoinGeckoClient
from src.api.dexscreener_client import DexScreenerClient
from src.api.blockchain_rpc import SolanaRPC, EtherscanClient

# Importar configuracion
import config

print("Clientes importados correctamente!")
print(f"Helius API key configurada: {'Si' if config.HELIUS_API_KEY else 'No (necesaria para Solana)'}")
print(f"Etherscan API key configurada: {'Si' if config.ETHERSCAN_API_KEY else 'No (necesaria para ETH)'}")
print(f"CoinGecko API key configurada: {'Si' if config.COINGECKO_API_KEY else 'No (opcional)'}")

---
## 1. GeckoTerminal API

API principal para datos on-chain. **Sin API key necesaria.**
- Rate limit: 30 calls/minuto
- Sin limite mensual documentado
- Datos: pools, precios, OHLCV, liquidez, volumen

In [ ]:
# Crear cliente de GeckoTerminal
gecko = CoinGeckoClient()
print(f"Cliente creado: {gecko}")

In [ ]:
# 1a. Descubrir pools nuevos en Solana (ultimas 48 horas)
# Esto es lo que usaremos para recopilacion prospectiva diaria
new_pools = gecko.get_new_pools(chain="solana", page=1)

if new_pools:
    print(f"Pools nuevos encontrados: {len(new_pools)}")
    print(f"\nEjemplo del primer pool:")
    # Mostrar las keys disponibles
    pool = new_pools[0]
    for key, value in pool.items():
        print(f"  {key}: {value}")
else:
    print("No se obtuvieron pools. Verifica tu conexion a internet.")

In [ ]:
# 1b. Obtener datos OHLCV (velas de precio) para un pool
# Usemos un pool conocido para probar
if new_pools:
    # Usar el primer pool que encontramos
    test_pool = new_pools[0]
    pool_address = test_pool.get("pool_address", "")
    print(f"Obteniendo OHLCV para: {test_pool.get('name', 'Unknown')}")
    print(f"Pool address: {pool_address}")
    
    ohlcv = gecko.get_pool_ohlcv(
        chain="solana",
        pool_address=pool_address,
        timeframe="hour",  # velas horarias
        limit=24,           # ultimas 24 horas
    )
    
    if ohlcv:
        print(f"\nVelas obtenidas: {len(ohlcv)}")
        print(f"Primera vela: {ohlcv[0]}")
        print(f"Ultima vela: {ohlcv[-1]}")
    else:
        print("No se obtuvieron datos OHLCV")
else:
    print("Primero ejecuta la celda anterior para obtener pools")

In [ ]:
# 1c. Buscar un token por nombre
results = gecko.search_pools(query="PEPE")

if results:
    print(f"Resultados de busqueda: {len(results)}")
    for i, pool in enumerate(results[:5]):  # Mostrar top 5
        print(f"\n--- Resultado {i+1} ---")
        print(f"  Nombre: {pool.get('name')}")
        print(f"  Pool: {pool.get('pool_address', '')[:20]}...")
        print(f"  Chain: {pool.get('chain')}")
else:
    print("No se encontraron resultados")

In [ ]:
# 1d. Pools trending en Solana
trending = gecko.get_trending_pools(chain="solana")

if trending:
    print(f"Pools trending: {len(trending)}")
    for i, pool in enumerate(trending[:5]):
        print(f"  {i+1}. {pool.get('name', 'Unknown')} - ${pool.get('price_usd', 'N/A')}")
else:
    print("No se obtuvieron trending pools")

---
## 2. DexScreener API

Fuente **unica** para buyers/sellers counts.
- Rate limit: 300 calls/minuto (muy generoso)
- Sin API key necesaria
- Datos: txns (buys/sells), price changes, boosts

In [ ]:
# Crear cliente de DexScreener
dex = DexScreenerClient()
print(f"Cliente creado: {dex}")

In [ ]:
# 2a. Buscar pares para un token conocido
# Usemos el address de algun token de los pools nuevos, o un token conocido
if new_pools:
    test_token = new_pools[0].get("base_token_address", "")
    if test_token:
        pairs = dex.get_token_pairs(chain="solana", token_address=test_token)
        
        if pairs:
            print(f"Pares encontrados: {len(pairs)}")
            pair = pairs[0]
            print(f"\nDatos del primer par:")
            for key, value in pair.items():
                # No imprimir valores muy largos
                val_str = str(value)
                if len(val_str) > 100:
                    val_str = val_str[:100] + "..."
                print(f"  {key}: {val_str}")
        else:
            print("No se encontraron pares en DexScreener")
else:
    print("Ejecuta las celdas de GeckoTerminal primero")

In [ ]:
# 2b. Tokens con boosts (señal social)
boosted = dex.get_boosted_tokens()

if boosted:
    print(f"Tokens boosted: {len(boosted)}")
    for i, token in enumerate(boosted[:5]):
        print(f"  {i+1}. {token}")
else:
    print("No se obtuvieron tokens boosted")

---
## 3. Solana RPC (Helius)

**Unica fuente gratuita** para datos de holders en Solana.
- Necesita API key de Helius (gratis, 1M creditos)
- Top 20 holders de cualquier token SPL
- Supply total y circulante

In [ ]:
# Crear cliente de Solana RPC
if config.HELIUS_API_KEY:
    sol_rpc = SolanaRPC()
    print(f"Cliente Solana RPC creado: {sol_rpc}")
else:
    print("⚠️ HELIUS_API_KEY no configurada en .env")
    print("Obtener gratis en: https://www.helius.dev/")
    sol_rpc = None

In [ ]:
# 3a. Obtener top holders de un token Solana
# Ejemplo: usar un token de los pools nuevos
if sol_rpc and new_pools:
    test_mint = new_pools[0].get("base_token_address", "")
    if test_mint:
        print(f"Consultando holders para: {test_mint[:20]}...")
        holders = sol_rpc.get_token_largest_accounts(test_mint)
        
        if holders:
            print(f"\nTop holders encontrados: {len(holders)}")
            for i, h in enumerate(holders[:5]):
                print(f"  #{i+1}: {h}")
        else:
            print("No se obtuvieron datos de holders")
elif not sol_rpc:
    print("Configura HELIUS_API_KEY primero")
else:
    print("Ejecuta las celdas de GeckoTerminal primero")

In [ ]:
# 3b. Obtener supply total de un token
if sol_rpc and new_pools:
    test_mint = new_pools[0].get("base_token_address", "")
    if test_mint:
        supply = sol_rpc.get_token_supply(test_mint)
        print(f"Supply del token: {supply}")
elif not sol_rpc:
    print("Configura HELIUS_API_KEY primero")

---
## 4. Etherscan API

Para verificacion de contratos en Ethereum y Base.
- Necesita API key (gratis, 100K calls/dia)
- Solo funciona para tokens ERC-20

In [ ]:
# Crear cliente de Etherscan
if config.ETHERSCAN_API_KEY:
    etherscan = EtherscanClient()
    print(f"Cliente Etherscan creado: {etherscan}")
else:
    print("⚠️ ETHERSCAN_API_KEY no configurada en .env")
    print("Obtener gratis en: https://etherscan.io/apis")
    etherscan = None

In [ ]:
# 4a. Verificar si un contrato esta verificado
# Ejemplo: PEPE token en Ethereum
PEPE_ADDRESS = "0x6982508145454Ce325dDbE47a25d4ec3d2311933"

if etherscan:
    is_verified = etherscan.is_contract_verified(PEPE_ADDRESS)
    print(f"PEPE contrato verificado: {is_verified}")
    
    # Obtener info del contrato
    source = etherscan.get_contract_source(PEPE_ADDRESS)
    if source:
        print(f"\nInfo del contrato:")
        for key, value in source.items():
            val_str = str(value)[:80]
            print(f"  {key}: {val_str}")
else:
    print("Configura ETHERSCAN_API_KEY primero")

---
## 5. CoinGecko Demo API

Para datos de contexto de mercado (BTC, ETH, SOL).
- 10,000 calls/mes (conservar cuota)
- API key opcional pero recomendada

In [ ]:
# 5a. Precio historico de Bitcoin (30 dias)
# Esto lo usaremos para el feature 'btc_return_7d_at_launch'
btc_history = gecko.get_coin_price_history(coin_id="bitcoin", days=30)

if btc_history:
    print(f"Datos de BTC obtenidos: {len(btc_history)} puntos")
    print(f"Primer punto: {btc_history[0]}")
    print(f"Ultimo punto: {btc_history[-1]}")
else:
    print("No se obtuvieron datos de BTC")

---
## 6. Resumen

### APIs probadas y estado:

| API | Estado | Datos clave |
|-----|--------|-------------|
| GeckoTerminal | ✅ / ❌ | OHLCV, pools, liquidez, volumen |
| DexScreener | ✅ / ❌ | Buyers/sellers, boosts |
| Solana RPC | ✅ / ❌ | Top 20 holders, supply |
| Etherscan | ✅ / ❌ | Verificacion contratos |
| CoinGecko | ✅ / ❌ | Precios BTC/ETH/SOL |

### Siguiente paso:
Ejecutar el notebook `02_data_collection.ipynb` para empezar a recopilar datos sistematicamente.

In [ ]:
# Resumen de llamadas realizadas
print("=" * 50)
print("RESUMEN DE LLAMADAS API")
print("=" * 50)
print(f"GeckoTerminal: {gecko.call_count} llamadas")
print(f"DexScreener:   {dex.call_count} llamadas")
if sol_rpc:
    print(f"Solana RPC:    {sol_rpc.call_count} llamadas")
if etherscan:
    print(f"Etherscan:     {etherscan.call_count} llamadas")